# Lab X: Deep Recurrent Q-Network (DRQN) on Atari Games

This tutorial shows how to train a Deep Recurrent Q-Network (DRQN) to play **Pong** and **Breakout** using frame sequences and LSTM for partial observability scenarios.

## ✅ Step 1: Install Required Packages

In [3]:
!pip install gymnasium[atari,accept-rom-license] opencv-python

## ✅ Step 2: Import Libraries

In [2]:
import gymnasium as gym
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from collections import deque
import random
import cv2
import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'tensorflow'

## ✅ Step 3: Frame Preprocessing and Stacking

In [ ]:
def preprocess_frame(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
    resized = cv2.resize(gray, (84, 84))
    normalized = resized / 255.0
    return normalized

class DRQNFrameStack:
    def __init__(self, seq_length=10):
        self.seq_length = seq_length
        self.frames = deque(maxlen=seq_length)

    def reset(self, frame):
        processed = preprocess_frame(frame)
        self.frames = deque([processed] * self.seq_length, maxlen=self.seq_length)
        return np.stack(self.frames, axis=0)

    def step(self, frame):
        processed = preprocess_frame(frame)
        self.frames.append(processed)
        return np.stack(self.frames, axis=0)

## ✅ Step 4: DRQN Model

In [ ]:
def build_drqn(input_shape, n_actions):
    inputs = layers.Input(shape=input_shape)  # (seq_len, 84, 84)
    x = layers.TimeDistributed(layers.Reshape((84, 84, 1)))(inputs)
    x = layers.TimeDistributed(layers.Conv2D(32, (8, 8), strides=4, activation='relu'))(x)
    x = layers.TimeDistributed(layers.Conv2D(64, (4, 4), strides=2, activation='relu'))(x)
    x = layers.TimeDistributed(layers.Conv2D(64, (3, 3), strides=1, activation='relu'))(x)
    x = layers.TimeDistributed(layers.Flatten())(x)
    x = layers.LSTM(256)(x)
    outputs = layers.Dense(n_actions, activation='linear')(x)
    model = models.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=optimizers.Adam(learning_rate=0.00025), loss='huber')
    return model

## ✅ Step 5: Training Setup

In [ ]:
# Choose one
GAME = "ALE/Pong-v5"  # or "ALE/Breakout-v5"

env = gym.make(GAME, render_mode=None)
n_actions = env.action_space.n
seq_len = 10
frame_stack = DRQNFrameStack(seq_len)

main_model = build_drqn((seq_len, 84, 84), n_actions)
target_model = build_drqn((seq_len, 84, 84), n_actions)
target_model.set_weights(main_model.get_weights())

memory = deque(maxlen=100000)

def remember(s, a, r, s_, done):
    memory.append((s, a, r, s_, done))

def act(state, epsilon):
    if np.random.rand() <= epsilon:
        return random.randrange(n_actions)
    q_vals = main_model.predict(state[np.newaxis], verbose=0)
    return np.argmax(q_vals[0])

def replay(batch_size, gamma):
    minibatch = random.sample(memory, batch_size)
    for state, action, reward, next_state, done in minibatch:
        target = main_model.predict(state[np.newaxis], verbose=0)
        next_q_main = main_model.predict(next_state[np.newaxis], verbose=0)
        next_q_target = target_model.predict(next_state[np.newaxis], verbose=0)
        best_action = np.argmax(next_q_main[0])
        if done:
            target[0][action] = reward
        else:
            target[0][action] = reward + gamma * next_q_target[0][best_action]
        main_model.fit(state[np.newaxis], target, epochs=1, verbose=0)

## ✅ Step 6: Training Loop

In [ ]:
episodes = 500
batch_size = 32
gamma = 0.99
epsilon = 1.0
epsilon_min = 0.1
epsilon_decay = 0.995
scores = []

for e in range(episodes):
    obs, _ = env.reset()
    state = frame_stack.reset(obs)
    done = False
    total_reward = 0

    while not done:
        action = act(state, epsilon)
        obs_, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        next_state = frame_stack.step(obs_)
        remember(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward

        if len(memory) > batch_size:
            replay(batch_size, gamma)

    if epsilon > epsilon_min:
        epsilon *= epsilon_decay

    target_model.set_weights(main_model.get_weights())
    scores.append(total_reward)
    print(f"Episode {e+1}, Score: {total_reward}, Epsilon: {epsilon:.3f}")

## ✅ Step 7: Results

In [ ]:
plt.plot(scores)
plt.title(f"DRQN - {GAME}")
plt.xlabel("Episode")
plt.ylabel("Total Reward")
plt.grid(True)
plt.show()

env.close()